In [1]:
import pandas as pd
from datetime import datetime
import pytz


In [2]:
df = pd.read_excel('CCTime_Zone.xlsx')
df.head(5)

,MeetingID,StartTime,EndTime,Country
0,1601,2022-01-01 08:00:00,2022-01-01 09:00:00,America/New_York
1,1602,2022-01-01 10:00:00,2022-01-01 11:00:00,Asia/Kolkata
2,1603,2022-01-02 12:00:00,2022-01-02 13:00:00,Europe/London
3,1604,2022-01-02 15:00:00,2022-01-02 16:00:00,Australia/Sydney
4,1605,2022-01-03 07:00:00,2022-01-03 08:00:00,America/Sao_Paulo


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10 entries, 0 to 9
Data columns (total 4 columns):
 #   Column     Non-Null Count  Dtype         
---  ------     --------------  -----         
 0   MeetingID  10 non-null     int64         
 1   StartTime  10 non-null     datetime64[ns]
 2   EndTime    10 non-null     datetime64[ns]
 3   Country    10 non-null     object        
dtypes: datetime64[ns](2), int64(1), object(1)
memory usage: 452.0+ bytes


In [4]:
# Function to convert time to UTC
def convert_to_utc(time_str, country):
    try:
        local = pytz.timezone(country)
    except pytz.UnknownTimeZoneError:
        print(f"Unknown time zone: {country}")
        return datetime.now()  # Default to current datetime
    if isinstance(time_str, pd.Timestamp):
        time_str = str(time_str)  # Convert Timestamp to string
    naive = datetime.strptime(time_str, '%Y-%m-%d %H:%M:%S')  # Updated format string
    local_dt = local.localize(naive, is_dst=None)
    utc_dt = local_dt.astimezone(pytz.utc)
    return utc_dt

In [5]:
# Convert start and end time to UTC
df['StartTime_UTC'] = df.apply(lambda x: convert_to_utc(x['StartTime'], x['Country']).strftime('%Y-%m-%d %H:%M:%S'), axis=1)
df['EndTime_UTC'] = df.apply(lambda x: convert_to_utc(x['EndTime'], x['Country']).strftime('%Y-%m-%d %H:%M:%S'), axis=1)

Unknown time zone:  Europe/London
Unknown time zone:  America/Sao_Paulo
Unknown time zone:  Asia/Tokyo
Unknown time zone:  Europe/Paris
Unknown time zone:  Europe/London
Unknown time zone:  America/Sao_Paulo
Unknown time zone:  Asia/Tokyo
Unknown time zone:  Europe/Paris


In [6]:
# Function to get timezone offset
def get_timezone_offset(country, start_time):
    try:
        timezone = pytz.timezone(country)
        offset = timezone.utcoffset(start_time)
        offset_hours = offset.total_seconds() // 3600
        offset_minutes = (offset.total_seconds() % 3600) // 60
        offset_str = f"UTC {'+' if offset_hours >= 0 else '-'}{abs(offset_hours)}:{abs(offset_minutes)}"
        return offset_str
    except pytz.UnknownTimeZoneError:
        print(f"Unknown time zone: {country}")
        return 'Unknown'

# Add timezone offset to the Country column
df['Timezone_Offset'] = df.apply(lambda row: get_timezone_offset(row['Country'], row['StartTime']), axis=1)

# Remove the decimal part from timezone offset
df['Timezone_Offset'] = df['Timezone_Offset'].str.replace('.0', '')

Unknown time zone:  Europe/London
Unknown time zone:  America/Sao_Paulo
Unknown time zone:  Asia/Tokyo
Unknown time zone:  Europe/Paris


In [7]:
df[['MeetingID', 'StartTime', 'EndTime', 'Country', 'StartTime_UTC', 'EndTime_UTC', 'Timezone_Offset']]

,MeetingID,StartTime,EndTime,Country,StartTime_UTC,EndTime_UTC,Timezone_Offset
0,1601,2022-01-01 08:00:00,2022-01-01 09:00:00,America/New_York,2022-01-01 13:00:00,2022-01-01 14:00:00,UTC -5:0
1,1602,2022-01-01 10:00:00,2022-01-01 11:00:00,Asia/Kolkata,2022-01-01 04:30:00,2022-01-01 05:30:00,UTC +5:30
2,1603,2022-01-02 12:00:00,2022-01-02 13:00:00,Europe/London,2024-02-12 14:46:41,2024-02-12 14:46:41,Unknown
3,1604,2022-01-02 15:00:00,2022-01-02 16:00:00,Australia/Sydney,2022-01-02 04:00:00,2022-01-02 05:00:00,UTC +11:0
4,1605,2022-01-03 07:00:00,2022-01-03 08:00:00,America/Sao_Paulo,2024-02-12 14:46:41,2024-02-12 14:46:41,Unknown
5,1606,2022-01-03 18:00:00,2022-01-03 19:00:00,Asia/Tokyo,2024-02-12 14:46:41,2024-02-12 14:46:41,Unknown
6,1607,2022-01-04 09:00:00,2022-01-04 10:00:00,America/Toronto,2022-01-04 14:00:00,2022-01-04 15:00:00,UTC -5:0
7,1608,2022-01-04 11:00:00,2022-01-04 12:00:00,Europe/Paris,2024-02-12 14:46:41,2024-02-12 14:46:41,Unknown
8,1609,2022-01-05 14:00:00,2022-01-05 15:00:00,Europe/Berlin,2022-01-05 13:00:00,2022-01-05 14:00:00,UTC +1:0
9,1610,2022-01-05 16:00:00,2022-01-05 17:00:00,Asia/Shanghai,2022-01-05 08:00:00,2022-01-05 09:00:00,UTC +8:0
